태블로용 시각화용 데이터 추출

In [2]:
# PostgreSQL에서 clean_orders 데이터 불러오기
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import urllib.parse
import matplotlib.pyplot as plt

load_dotenv('.env')  # .env 파일 읽기

# DB연결정보
user = "postgres"
password = urllib.parse.quote_plus(os.getenv('DB_PW'))
host = "127.0.0.1"
port = '5432'
database = 'ecommerce'

# 엔진 생성
engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{database}')

# 데이터 불러오기
data = pd.read_sql("select * from clean_orders", engine)

# 데이터 전처리
order_completed = data[data['order_status']=='Delivered to buyer']
df = order_completed.dropna(subset=['item_total'])

In [6]:
# 월별 매출
df['year_month'] = df['order_date'].dt.to_period('M')
month_sales = df.groupby('year_month')['item_total'].sum().reset_index()

month_sales.to_csv('month_sales.csv', index=False)

# 재구매 여부
repeat_customer = df.groupby('buyer').size().reset_index(name='purchase_count')
repeat_customers = repeat_customer[repeat_customer['purchase_count'] >= 2]

repeat_customers.to_csv('repeat_customers.csv', index=False)

# 고객 매출
customer_sales = df.groupby('buyer')['item_total'].sum().reset_index()
customer_sales.to_csv('customer_sales.csv', index=False)

C:\Users\82104\AppData\Local\Temp\ipykernel_21940\1927919121.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['year_month'] = df['order_date'].dt.to_period('M')
